# 04 - Evaluate and promote

Build plane, `evaluate` task of the `msd_training` job.

* Loads the exact model version registered by `02_train` (task value `model_version`, or the
  `model_version` parameter for manual runs) from Unity Catalog.
* Scores the held-out **test** split (30 images per class) through the same pyfunc interface
  the serving endpoint uses (base64 image in).
* Computes overall accuracy and per-class precision, recall, F1 and support, plus the confusion
  matrix; logs them to the version's MLflow run and appends them to `model_eval_metrics`.
* **Promotion gate**: sets the `champion` alias on this version only if test accuracy is at
  least `min_test_accuracy` (default 0.90). Otherwise the previous champion stays in place.

In [0]:
%pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cpu
%restart_python

In [0]:
dbutils.widgets.text("catalog", "poc_metal_surface_defect_diagnostics")
dbutils.widgets.text("schema", "metal_surface_defects")
dbutils.widgets.text("model_name", "msd_defect_classifier")
dbutils.widgets.text("model_version", "")  # empty -> task value from the train task
dbutils.widgets.text("min_test_accuracy", "0.90")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
registered_model_name = f"{catalog}.{schema}.{dbutils.widgets.get('model_name')}"
min_test_accuracy = float(dbutils.widgets.get("min_test_accuracy"))
model_version = dbutils.widgets.get("model_version") or dbutils.jobs.taskValues.get(
    taskKey="train", key="model_version", default="", debugValue=""
)
if not model_version:
    raise ValueError("No model_version: pass the parameter or run after the train task.")

metrics_table = f"{catalog}.{schema}.model_eval_metrics"
index_table = f"{catalog}.{schema}.image_index"
bronze_table = f"{catalog}.{schema}.image_bronze"
CHAMPION = "champion"
print(f"evaluating {registered_model_name} v{model_version} (gate {min_test_accuracy:.0%})")

In [0]:
import base64
import json

import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")
model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}/{model_version}")

test = spark.sql(f"""
    SELECT i.file_name, i.label, b.content
    FROM {index_table} i
    JOIN {bronze_table} b ON i.path = regexp_replace(b.path, '^dbfs:', '')
    WHERE i.split = 'test'
""").toPandas()

preds = model.predict(pd.DataFrame({"image": [base64.b64encode(b).decode() for b in test["content"]]}))
results = pd.concat([test[["file_name", "label"]].reset_index(drop=True), preds], axis=1)
print(f"scored {len(results)} test images")

In [0]:
classes = sorted(json.loads(results["probabilities"].iloc[0]).keys())
accuracy = float((results["label"] == results["predicted_class"]).mean())

per_class = []
for c in classes:
    tp = int(((results.label == c) & (results.predicted_class == c)).sum())
    fp = int(((results.label != c) & (results.predicted_class == c)).sum())
    fn = int(((results.label == c) & (results.predicted_class != c)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    per_class.append({"defect_class": c, "precision": precision, "recall": recall, "f1": f1, "support": tp + fn})
per_class_df = pd.DataFrame(per_class)
macro_f1 = float(per_class_df["f1"].mean())

confusion = pd.crosstab(results["label"], results["predicted_class"], rownames=["actual"], colnames=["predicted"]).reindex(
    index=classes, columns=classes, fill_value=0
)
print(f"test accuracy {accuracy:.4f}, macro F1 {macro_f1:.4f}")
display(per_class_df)
display(confusion.reset_index())

In [0]:
from datetime import datetime, timezone

from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
version_info = client.get_model_version(registered_model_name, str(model_version))
run_id = version_info.run_id

tracking = MlflowClient()
tracking.log_metric(run_id, "test_accuracy", accuracy)
tracking.log_metric(run_id, "test_macro_f1", macro_f1)
for r in per_class:
    tracking.log_metric(run_id, f"test_f1_{r['defect_class']}", r["f1"])
tracking.log_text(run_id, confusion.to_csv(), "evaluation/confusion_matrix.csv")
tracking.log_text(run_id, per_class_df.to_csv(index=False), "evaluation/per_class_metrics.csv")

evaluated_at = datetime.now(timezone.utc)
rows = [
    (str(model_version), r["defect_class"], float(r["precision"]), float(r["recall"]), float(r["f1"]),
     int(r["support"]), accuracy, macro_f1, evaluated_at)
    for r in per_class
]
metrics_df = spark.createDataFrame(
    rows,
    "model_version string, defect_class string, precision double, recall double, f1 double, "
    "support int, accuracy double, macro_f1 double, evaluated_at timestamp",
)
metrics_df.write.mode("append").saveAsTable(metrics_table)
spark.sql(
    f"COMMENT ON TABLE {metrics_table} IS 'Test-split evaluation per model version and defect class "
    "(appended by 04_evaluate)'"
)
print(f"logged to run {run_id} and appended {len(rows)} rows to {metrics_table}")

In [0]:
promoted = accuracy >= min_test_accuracy
if promoted:
    client.set_registered_model_alias(registered_model_name, CHAMPION, str(model_version))
    client.set_model_version_tag(registered_model_name, str(model_version), "test_accuracy", f"{accuracy:.4f}")
    print(f"PROMOTED: v{model_version} is now @{CHAMPION} (test accuracy {accuracy:.2%})")
else:
    print(f"NOT PROMOTED: test accuracy {accuracy:.2%} < {min_test_accuracy:.0%}; @{CHAMPION} unchanged")

dbutils.jobs.taskValues.set(key="promoted", value=bool(promoted))
dbutils.jobs.taskValues.set(key="test_accuracy", value=accuracy)